In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import rankdata

FILE = '../ha-2sec-full-rl-v4.pqt'
L_PTS = 32
MIN_LEN = 4
MIN_RANGE_PTS = 0.5
TICK = 0.0625
POOL_CAP = 600_000
PAIR_CAP = 150_000
TEST_FRAC = 0.2
NUM_ROUNDS = 150
PARAMS = {
    'objective': 'binary',
    'learning_rate': 0.1,
    'num_leaves': 63,
    'min_data_in_leaf': 100,
    'verbose': -1,
    'seed': 7,
}

rng = np.random.default_rng(7)

df = pd.read_parquet(FILE, columns=['date', 'Close', 'jmaD1'])
date = df['date'].to_numpy()
close = df['Close'].to_numpy(np.float64)
s = pd.Series(np.sign(df['jmaD1'].to_numpy())).replace(0.0, np.nan)
s = s.groupby(date).ffill().groupby(date).bfill().to_numpy()
m = len(df)
new_day = np.empty(m, dtype=bool); new_day[0] = True
new_day[1:] = date[1:] != date[:-1]
flip = np.empty(m, dtype=bool); flip[0] = True
flip[1:] = s[1:] != s[:-1]
starts = np.flatnonzero(new_day | flip)
ends = np.r_[starts[1:], m]
lens = ends - starts
keep = np.flatnonzero(lens >= MIN_LEN)
if len(keep) > POOL_CAP:
    keep = rng.choice(keep, POOL_CAP, replace=False)
xs = np.linspace(0.0, 1.0, L_PTS)
shapes = np.empty((len(keep), L_PTS), np.float32)
rng_pts = np.empty(len(keep), np.float64)
days = date[starts[keep]]
seg_len = lens[keep]
ok = np.ones(len(keep), dtype=bool)
for i, k in enumerate(keep):
    seg = close[starts[k]:ends[k]]
    lo = seg.min()
    r = seg.max() - lo
    if r < MIN_RANGE_PTS:
        ok[i] = False
        continue
    shapes[i] = np.interp(xs, np.linspace(0.0, 1.0, lens[k]), seg - lo) / r
    rng_pts[i] = r
shapes, rng_pts, days, seg_len = shapes[ok], rng_pts[ok], days[ok], seg_len[ok]
print(f"segments kept: {len(shapes):,}", flush=True)

day_med = pd.Series(rng_pts).groupby(days).median()
q25, q75 = day_med.quantile(0.25), day_med.quantile(0.75)
lo_days = set(day_med.index[day_med <= q25])
hi_days = set(day_med.index[day_med >= q75])
cls_lo = np.array([d in lo_days for d in days])
cls_hi = np.array([d in hi_days for d in days])
print(f"quiet-quartile days: {len(lo_days)}  loud-quartile days: {len(hi_days)}")

bins = np.minimum(seg_len, 48) * 1000 + np.floor(np.log2(rng_pts / TICK) * 4.0).astype(np.int64)
ia_l, ib_l = [], []
idx_lo = np.flatnonzero(cls_lo)
idx_hi = np.flatnonzero(cls_hi)
b_lo = bins[idx_lo]
b_hi = bins[idx_hi]
for b in np.intersect1d(np.unique(b_lo), np.unique(b_hi)):
    ca = idx_lo[b_lo == b]
    cb = idx_hi[b_hi == b]
    k = min(len(ca), len(cb))
    if k:
        ia_l.append(rng.choice(ca, k, replace=False))
        ib_l.append(rng.choice(cb, k, replace=False))
ia = np.concatenate(ia_l)
ib = np.concatenate(ib_l)
if len(ia) > PAIR_CAP:
    sel = rng.choice(len(ia), PAIR_CAP, replace=False)
    ia, ib = ia[sel], ib[sel]
take = len(ia)
X = np.vstack([shapes[ia], shapes[ib]])
y = np.r_[np.zeros(take, np.int8), np.ones(take, np.int8)]
day = np.concatenate([days[ia], days[ib]])
ud = np.unique(day)
te_days = rng.choice(ud, int(len(ud) * TEST_FRAC), replace=False)
te = np.isin(day, te_days)
tr = ~te
mdl = lgb.train(PARAMS, lgb.Dataset(X[tr], label=y[tr]), num_boost_round=NUM_ROUNDS)
p = mdl.predict(X[te])
r_ = rankdata(p)
n1 = int(y[te].sum())
n0 = int((~te).size and len(y[te]) - n1)
auc = (r_[y[te] == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0)
print(f"within-frame day-climate AUC at matched (L, range): {auc:.4f}  (n/class={take:,})")

segments kept: 593,494
quiet-quartile days: 297  loud-quartile days: 300
within-frame day-climate AUC at matched (L, range): 0.6831  (n/class=72,829)
